# Notebook Setup

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

# Portfolio Setup

In [2]:
tickers = ['AAPL', 'TSLA', 'MSFT', 'META', 'AMZN', 'GOOGL', 'NVDA']  # Ticker der MAG7
benchmark_world_ticker = 'VT'
benchmark_risk_free_ticker = '^TNX'
benchmark_sp500_ticker = '^SPX'
benchmark_nasdaq_ticker = '^NDX'

start_capital = 100_000  # USD
start_date = "2012-05-18"  # gemeinsamer Startpunkt ab Meta-IPO
end_date = "2026-04-01"

confidence_levels = {
    "95 %": 0.05,
    "99 %": 0.01,
}
reference_alpha = confidence_levels["95 %"]

horizons = {
    "1 Jahr": 252,
    "5 Jahre": 1260,
    "10 Jahre": 2520,
}

monte_carlo_simulations = 10_000
bootstrap_simulations = 10_000
monte_carlo_seed = 2
bootstrap_seed = 1

In [3]:
data = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    auto_adjust=False,
    progress=False,
)['Adj Close']

In [4]:
benchmark_world_data = yf.download(
    benchmark_world_ticker,
    start=start_date,
    end=end_date,
    auto_adjust=False,
    progress=False,
)['Adj Close']  # VT (Vanguard Total World Stock ETF)

benchmark_risk_free_data = yf.download(
    benchmark_risk_free_ticker,
    start=start_date,
    end=end_date,
    auto_adjust=False,
    progress=False,
)['Adj Close']  # ^TNX (10-Year Treasury Note Yield)

benchmark_sp500_data = yf.download(
    benchmark_sp500_ticker,
    start=start_date,
    end=end_date,
    auto_adjust=False,
    progress=False,
)['Adj Close'] 

benchmark_nasdaq_data = yf.download(
    benchmark_nasdaq_ticker,
    start=start_date,
    end=end_date,
    auto_adjust=False,
    progress=False,
)['Adj Close'] 

In [5]:
dotcom_data = yf.download(
    '^NDX',  # Nasdaq-ETF als Proxy für Dotcom-Blase
    start="2000-04-01",
    end="2002-10-31",
    auto_adjust=False,
    progress=False,
)['Adj Close'] 

dotcom_log_returns = np.log(dotcom_data / dotcom_data.shift(1)).dropna().values

In [6]:
#Berechnung der täglichen Renditen (diskret und logarithmisch)
returns_discrete = data.pct_change().dropna() #berechnet die prozentuale Veränderung der Renditen - diskret
returns_log = np.log(data / data.shift(1)).dropna() #berechnet die logarithmische Rendite

#Portfolio-Gewichtung
weights = np.array([1/len(tickers)] * len(tickers)) #1/7 Gewichtung für jedes Asset

#Berechnung der Portfolio-Renditen
portfolio_returns_discrete = returns_discrete.dot(weights) #Berechnung der Portfolio-Renditen
portfolio_returns_log = returns_log.dot(weights)

Wir berechnen sowohl **diskrete Renditen** als auch **Log-Renditen**.

- **Diskrete Renditen** sind intuitiv für Prozentveränderungen von einem Tag auf den nächsten.
- **Log-Renditen** sind über die Zeit additiv und deshalb besonders nützlich für Mehrtageshorizonte, historische Aggregation und Monte-Carlo-Simulation.

Das Portfolio wird als **gleichgewichtet** modelliert, also mit `1/7` pro Aktie.

In [7]:
portfolio_returns_discrete.tail()

Date
2026-03-25    0.007628
2026-03-26   -0.031970
2026-03-27   -0.027635
2026-03-30   -0.001346
2026-03-31    0.045284
dtype: float64

In [8]:
def plot_historical_performance(portfolio_returns, market_returns, sp500_returns, nasdaq_returns, start_capital=100000, title="Historische Renditeentwicklung"):
    """
    Erstellt ein Liniendiagramm, das die historische Entwicklung von Portfolio, 
    Markt und risikofreiem Zins (auf Basis des Startkapitals) vergleicht.
    """
    # 1. Daten synchronisieren (wie bei den KPIs)
    aligned_data = pd.concat([
        portfolio_returns.squeeze().rename('Portfolio'), 
        market_returns.squeeze().rename('Market'), 
        sp500_returns.squeeze().rename('SP500'),
        nasdaq_returns.squeeze().rename('NASDAQ'),
    ], axis=1, sort=False).dropna()

    # 2. Renditen in kumulatives Wachstum umrechnen (Zinseszinseffekt)
    # Formel: Startkapital * kumuliertes Produkt von (1 + Rendite)
    portfolio_growth = start_capital * (1 + aligned_data['Portfolio']).cumprod()
    market_growth = start_capital * (1 + aligned_data['Market']).cumprod()
    sp500_growth = start_capital * (1 + aligned_data['SP500']).cumprod()
    nasdaq_growth = start_capital * (1 + aligned_data['NASDAQ']).cumprod()

    # Wir fügen Tag 0 (Startkapital) ganz vorne an, damit alle Kurven exakt im selben Punkt starten
    portfolio_growth.loc[aligned_data.index[0] - pd.Timedelta(days=1)] = start_capital
    market_growth.loc[aligned_data.index[0] - pd.Timedelta(days=1)] = start_capital
    sp500_growth.loc[aligned_data.index[0] - pd.Timedelta(days=1)] = start_capital
    nasdaq_growth.loc[aligned_data.index[0] - pd.Timedelta(days=1)] = start_capital
    
    # Sortieren, damit Tag 0 wirklich am Anfang steht
    portfolio_growth = portfolio_growth.sort_index()
    market_growth = market_growth.sort_index()
    sp500_growth = sp500_growth.sort_index()
    nasdaq_growth = nasdaq_growth.sort_index()

    fig = go.Figure()

    # Portfolio Linie (Mag7)
    fig.add_trace(go.Scatter(
        x=portfolio_growth.index, y=portfolio_growth,
        mode='lines', line=dict(color='rgb(52, 152, 219)', width=2), # Blau
        name='MAG7 Portfolio'
    ))

    # Benchmark Linie (Markt)
    fig.add_trace(go.Scatter(
        x=market_growth.index, y=market_growth,
        mode='lines', line=dict(color='rgb(231, 76, 60)', width=2), # Rot
        name='Markt (All-World-ETF)'
    ))

    # SP500 Linie
    fig.add_trace(go.Scatter(
        x=sp500_growth.index, y=sp500_growth,
        mode='lines', line=dict(color='rgb(142, 68, 173)', width=2), # Lila
        name='S&P 500'
    ))

    # NASDAQ Linie
    fig.add_trace(go.Scatter(
        x=nasdaq_growth.index, y=nasdaq_growth,
        mode='lines', line=dict(color='rgb(241, 196, 15)', width=2), # Gelb
        name='NASDAQ'
    ))

    # Layout optimieren
    fig.update_layout(
        title=title,
        xaxis_title="Datum",
        yaxis_title="Portfoliowert ($)",
        template="plotly_dark",
        hovermode="x unified",
        margin=dict(l=20, r=20, t=50, b=20),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )

    return fig

In [9]:
# 1. Funktion aufrufen und mit deinen echten Daten füttern
fig_hist = plot_historical_performance(
    portfolio_returns = portfolio_returns_discrete,
    market_returns = benchmark_world_data.pct_change().dropna(), 
    sp500_returns = benchmark_sp500_data.pct_change().dropna(),
    nasdaq_returns = benchmark_nasdaq_data.pct_change().dropna(),
    start_capital = 100000,
    title = "MAG7 vs. Markt vs. S&P 500 vs. NASDAQ: Historische Entwicklung"
)

# 2. Grafik zeichnen lassen
fig_hist.show()

## Verteilungsdiagnose der Mag7-Renditen

Bevor wir die parametrischen Methoden ernst nehmen, schauen wir kurz auf die Annahme: Gauß und Lognormal stehen und fallen damit, dass die Tagesrenditen halbwegs normal sind. Wir berichten Schiefe, Excess-Kurtosis und einen Jarque-Bera-Test, plus einen QQ-Plot gegen die Normalverteilung. Wenn die Punkte in den Tails von der 45-Grad-Linie abkippen, wissen wir, warum die Methoden gleich auseinanderlaufen.


In [10]:
# Kennzahlen der Tagesrenditeverteilung
clean_log_returns = portfolio_returns_log.dropna()

mu_d = clean_log_returns.mean()
sigma_d = clean_log_returns.std()
skew_d = stats.skew(clean_log_returns)
exkurt_d = stats.kurtosis(clean_log_returns)  # Fisher-Definition: Excess
jb_stat, jb_p = stats.jarque_bera(clean_log_returns)

distribution_diagnostics = pd.DataFrame({
    'Kennzahl': ['Mittelwert (taeglich)', 'Volatilitaet (taeglich)', 'Schiefe', 'Excess-Kurtosis', 'Jarque-Bera-Statistik', 'Jarque-Bera p-Wert'],
    'Wert': [mu_d, sigma_d, skew_d, exkurt_d, jb_stat, jb_p],
})
distribution_diagnostics['Wert'] = distribution_diagnostics['Wert'].round(6)
display(distribution_diagnostics)


,Kennzahl,Wert
0,Mittelwert (taeglich),0.001063
1,Volatilitaet (taeglich),0.016878
2,Schiefe,-0.359175
3,Excess-Kurtosis,5.553256
4,Jarque-Bera-Statistik,4552.961088
5,Jarque-Bera p-Wert,0.000000


In [11]:
# QQ-Plot gegen die Normalverteilung
sample = clean_log_returns.values
sample_sorted = np.sort(sample)
n = len(sample_sorted)
theoretical_q = stats.norm.ppf((np.arange(1, n + 1) - 0.5) / n)  # Plotting positions

# Referenzlinie über Mittelwert + Std der Stichprobe
line_x = np.array([theoretical_q.min(), theoretical_q.max()])
line_y = mu_d + sigma_d * line_x

fig_qq = go.Figure()
fig_qq.add_trace(go.Scatter(
    x=theoretical_q, y=sample_sorted,
    mode='markers',
    marker=dict(color='rgb(52, 152, 219)', size=4, opacity=0.6),
    name='Empirische Quantile',
))
fig_qq.add_trace(go.Scatter(
    x=line_x, y=line_y,
    mode='lines',
    line=dict(color='rgb(231, 76, 60)', width=2, dash='dash'),
    name='Normal-Referenz',
))
fig_qq.update_layout(
    template='plotly_dark',
    title='QQ-Plot: Mag7-Tagesrenditen vs. Normalverteilung',
    xaxis_title='Theoretisches Quantil (Standardnormalverteilung)',
    yaxis_title='Empirisches Quantil (Log-Renditen)',
    height=500,
)
fig_qq.show()


## Korrelationen innerhalb des Portfolios

Die Mag7 sind alle US-Tech-Schwergewichte und bewegen sich oft im Gleichschritt. Das ist relevant, weil hohe Korrelationen den Diversifikationseffekt im Portfolio kleiner machen, als die sieben Namen vermuten lassen. Die Heatmap zeigt die paarweisen Korrelationen der Tagesrenditen.


In [12]:
# Korrelationsmatrix der Mag7-Tagesrenditen
corr = returns_log.corr().round(2)
labels = list(corr.columns)
z = corr.values

fig_corr = go.Figure(data=go.Heatmap(
    z=z,
    x=labels,
    y=labels,
    colorscale='RdBu_r',
    zmin=-1, zmax=1,
    colorbar=dict(title='Korrelation'),
))

# Werte als Annotations darüber legen
annotations = []
for i, row_label in enumerate(labels):
    for j, col_label in enumerate(labels):
        annotations.append(dict(
            x=col_label, y=row_label,
            text=f"{z[i, j]:.2f}",
            showarrow=False,
            font=dict(color='white', size=12),
        ))

fig_corr.update_layout(
    template='plotly_dark',
    title='Korrelationsmatrix der Mag7-Tagesrenditen',
    annotations=annotations,
    width=650, height=600,
    yaxis=dict(autorange='reversed'),
)
fig_corr.show()


# Erstellung der Funktionen

## Berechnung der 1-Tages-Risikomodelle

Value at Risk und Expected Shortfall für historische, gaußsche und lognormale Modellierung. Der **historische VaR** liest direkt das empirische Quantil der Verlustverteilung ab, der **ES** mittelt über die Tail-Verluste jenseits des Quantils (Albrecht/Huggenberger 2015, S. 135 f.).


Wir berichten **VaR** und **Expected Shortfall** in der Profit-and-Loss-Konvention.

- **Negativer Wert:** im betrachteten Quantil liegt der Endwert unter dem Anfangskapital, also Verlust.
- **Positiver Wert:** selbst im betrachteten Quantil liegt der Endwert über dem Anfangskapital, also Gewinn.

Für die Normalmethode basiert der VaR auf dem linken Renditequantil der geschätzten Verteilung. Der **ES** mittelt über genau das Tail jenseits des VaR-Quantils und liegt deshalb auf gleichem Konfidenzniveau immer mindestens so tief wie der VaR. In P&L-Konvention gilt also `ES ≤ VaR` (Albrecht/Huggenberger 2015, S. 72 f.).


In [13]:
# 1. Historisches Risiko (VaR & ES)
def calculate_historical_risk_1d(returns, capital, var_level):
    alpha = var_level
    clean_returns = returns.dropna()

    var_pct = np.percentile(clean_returns, alpha * 100) # Berechnung des VaR als %-Quantil der Renditen
    tail_returns = clean_returns[clean_returns <= var_pct] #
    es_pct = np.mean(tail_returns)

    return capital * var_pct, capital * es_pct


# 2. Gaußsches Risiko (VaR & ES)
def calculate_gaussian_risk_1d(returns, capital, var_level):
    alpha = var_level
    clean_returns = returns.dropna()

    mu = np.mean(clean_returns) #Erwartungswert der Renditen
    sigma = np.std(clean_returns) #Volatilität der Renditen
    z_score = stats.norm.ppf(alpha) #z-Score für das Quantil, das dem VaR-Level entspricht

    var_pct = mu + (z_score * sigma) 
    es_pct = mu - sigma * (stats.norm.pdf(z_score) / alpha)

    return capital * var_pct, capital * es_pct


# 3. Lognormales Risiko (VaR & ES)
def calculate_lognormal_risk_1d(log_returns, capital, var_level):
    alpha = var_level
    clean_returns = log_returns.dropna()

    mu_log = np.mean(clean_returns)
    sigma_log = np.std(clean_returns)
    z_score = stats.norm.ppf(alpha)

    log_worst_case = mu_log + (z_score * sigma_log) #Renditen im logarithmischen Raum
    var_pct = np.exp(log_worst_case) - 1 #Umrechnung zurück in diskrete Renditen

    expected_value_lognorm = np.exp(mu_log + (sigma_log**2) / 2) #Erwartungswert der lognormalverteilten Renditen
    tail_factor = stats.norm.cdf(z_score - sigma_log) / alpha #Wahrscheinlichkeit, dass die Renditen im logarithmischen Raum unter dem Worst-Case liegen
    es_pct = (expected_value_lognorm * tail_factor) - 1

    return capital * var_pct, capital * es_pct

## VaR-Dichteplot mit Normal-Overlay - 1 Jahr

Wir legen über das gleiche empirische 1-Jahres-P&L die Normaldichte mit denselben Momenten (mu*252, sigma*sqrt(252)). So sieht man direkt, was die Risk-Folien (Schwarz S. 15/25) andeuten: die Normalverteilung ist im linken Tail systematisch zu duenn, die historischen 95-/99-VaR-Linien liegen weiter draußen als die Normalannahme suggeriert.


In [14]:
# Plot A: empirische 1J-Dichte vs. Normal-Overlay, mit historischen VaR-Linien
days_1y = horizons['1 Jahr']

# Empirische rollierende 1J-P&L (gleiche Datenbasis wie Plot 1)
rolling_log_1y_A = portfolio_returns_log.rolling(window=days_1y).sum().dropna()
rolling_pnl_1y_A = (np.exp(rolling_log_1y_A) - 1) * start_capital

# Normal-Overlay: Mu und Sigma der Tagesrenditen auf 1 Jahr skaliert,
# dann von Log-Raum in P&L-Raum gemappt über die Lognormal-Endwertformel.
clean_log = portfolio_returns_log.dropna()
mu_d_A = clean_log.mean()
sigma_d_A = clean_log.std()
mu_1y_A = mu_d_A * days_1y
sigma_1y_A = sigma_d_A * np.sqrt(days_1y)

# x-Grid über den empirischen Bereich
x_min = float(rolling_pnl_1y_A.min())
x_max = float(rolling_pnl_1y_A.max())
x_grid = np.linspace(x_min, x_max, 400)

# Normaldichte direkt auf der Log-Rendite, danach Variablentransformation
# x = (exp(r) - 1) * K -> r = log(1 + x/K), dr/dx = 1 / (K + x)
log_grid = np.log(1 + x_grid / start_capital)
normal_density_pnl = stats.norm.pdf(log_grid, loc=mu_1y_A, scale=sigma_1y_A) / (start_capital + x_grid)

# Historische VaR-Linien (95 % und 99 %), direkt aus der rollierenden 1J-P&L,
# weil calculate_historical_risk erst weiter unten definiert wird.
var_hist_95 = float(np.percentile(rolling_pnl_1y_A, 5))
var_hist_99 = float(np.percentile(rolling_pnl_1y_A, 1))

fig_pA = go.Figure()
fig_pA.add_trace(go.Histogram(
    x=rolling_pnl_1y_A,
    nbinsx=80,
    histnorm='probability density',
    marker=dict(color='rgb(52, 152, 219)', line=dict(color='rgb(30, 90, 130)', width=0.5)),
    name='Empirische Dichte',
    opacity=0.85,
))
fig_pA.add_trace(go.Scatter(
    x=x_grid, y=normal_density_pnl,
    mode='lines',
    line=dict(color='rgb(243, 156, 18)', width=2.5),
    name='Normal-Overlay',
))
fig_pA.add_vline(x=var_hist_95, line=dict(color='rgb(231, 76, 60)', width=2, dash='dash'),
                 annotation_text=f"VaR 95 %: {var_hist_95:,.0f} USD", annotation_position='top')
fig_pA.add_vline(x=var_hist_99, line=dict(color='rgb(231, 76, 60)', width=2, dash='dot'),
                 annotation_text=f"VaR 99 %: {var_hist_99:,.0f} USD", annotation_position='bottom')

fig_pA.update_layout(
    template='plotly_dark',
    title='VaR im Dichtevergleich - Empirisch vs. Normal - 1 Jahr',
    xaxis_title='Gewinn/Verlust (USD)',
    yaxis_title='Dichte',
    height=500,
    bargap=0.02,
)
fig_pA.show()


In [15]:
# 1-Tages-Blick als Kurzfrist-Referenz (95 %)
one_day_rows = []

hist_var, hist_es = calculate_historical_risk_1d(portfolio_returns_discrete, start_capital, reference_alpha)
one_day_rows.append({'Methode': 'Historisch', 'VaR ($)': hist_var, 'ES ($)': hist_es})

gauss_var, gauss_es = calculate_gaussian_risk_1d(portfolio_returns_discrete, start_capital, reference_alpha)
one_day_rows.append({'Methode': 'Gaußsch', 'VaR ($)': gauss_var, 'ES ($)': gauss_es})

lognorm_var, lognorm_es = calculate_lognormal_risk_1d(portfolio_returns_log, start_capital, reference_alpha)
one_day_rows.append({'Methode': 'Lognormal', 'VaR ($)': lognorm_var, 'ES ($)': lognorm_es})

one_day_results = pd.DataFrame(one_day_rows)
one_day_results['VaR (%)'] = one_day_results['VaR ($)'] / start_capital * 100
one_day_results['ES (%)'] = one_day_results['ES ($)'] / start_capital * 100

display(one_day_results.round(2))

,Methode,VaR ($),ES ($),VaR (%),ES (%)
0,Historisch,-2684.28,-3936.03,-2.68,-3.94
1,Gaußsch,-2640.08,-3345.08,-2.64,-3.35
2,Lognormal,-2634.08,-3316.34,-2.63,-3.32


## Skalierung auf Zeiträume: 1 Jahr, 5 Jahre und 10 Jahre

- **Historisch:** Für 1 Jahr nutzen wir rollierende historische Fenster. Für 5 und 10 Jahre nutzen wir historisches Bootstrapping, weil es für direkte langfristige Quantile zu wenige echte nicht überlappende Beobachtungen gibt.
- **Gaußsch:** Die Methode kann für alle Horizonte gerechnet werden. Für lange Horizonte wird sie aber annahmestärker, weil Mittelwert linear mit der Zeit und Volatilität mit der Wurzel aus der Zeit skaliert wird.
- **Lognormal / Monte Carlo:** Die Methode erzeugt viele Zukunftspfade auf Basis normalverteilter Log-Renditen. Das ist für längere Horizonte oft anschaulicher als eine reine Skalierung.

In [16]:
def calculate_historical_risk(log_returns, capital, var_level, days):
    if days == 1:
        rolling_discrete = np.exp(log_returns.dropna()) - 1 #keine Aggregation, da nur 1 Tag betrachtet wird
    else:
        rolling_log = log_returns.rolling(window=days).sum().dropna() # Rolling Window und Aggregation

        if len(rolling_log) < 100: #Sicherheitscheck: nicht weniger als 100 Beobachtungen für die Berechnung
            return np.nan, np.nan
        rolling_discrete = np.exp(rolling_log) - 1 #Log zu diskret

    var_pct = np.percentile(rolling_discrete, var_level * 100) 
    tail_returns = rolling_discrete[rolling_discrete <= var_pct]
    es_pct = np.mean(tail_returns)

    var_end_value = capital * (1 + var_pct)
    es_end_value = capital * (1 + es_pct)

    # 3. PnL-Logik (Endwert - Startkapital)
    var_pnl = var_end_value - capital
    es_pnl = es_end_value - capital

    return var_pnl, es_pnl

In [17]:
def calculate_gaussian_risk(returns, capital, var_level, days):
    clean_returns = returns.dropna()

    mu_1d = np.mean(clean_returns) #Erwartungswert der täglichen Renditen
    sigma_1d = np.std(clean_returns) #Volatilität der täglichen Renditen

    mu_nd = mu_1d * days 
    sigma_nd = sigma_1d * np.sqrt(days) #Skalierung der Volatilität mit der Quadratwurzel der Zeit

    z_score = stats.norm.ppf(var_level)

    var_pct = mu_nd + (z_score * sigma_nd)
    es_pct = mu_nd - sigma_nd * (stats.norm.pdf(z_score) / var_level)

    var_end_value = capital * (1 + var_pct)
    es_end_value = capital * (1 + es_pct)

    # 3. PnL-Logik (Endwert - Startkapital)
    var_pnl = var_end_value - capital
    es_pnl = es_end_value - capital

    return var_pnl, es_pnl

In [18]:
def calculate_lognormal_risk(log_returns, capital, var_level, days):
    clean_returns = log_returns.dropna()

    mu_log_1d = np.mean(clean_returns)
    sigma_log_1d = np.std(clean_returns)

    mu_log_nd = mu_log_1d * days
    sigma_log_nd = sigma_log_1d * np.sqrt(days)

    z_score = stats.norm.ppf(var_level)

    log_worst_case = mu_log_nd + (z_score * sigma_log_nd)
    var_end_value = capital * np.exp(log_worst_case)

    expected_value_lognorm = np.exp(mu_log_nd + (sigma_log_nd**2) / 2)
    tail_factor = stats.norm.cdf(z_score - sigma_log_nd) / var_level
    es_end_value = capital * expected_value_lognorm * tail_factor

    # 2. PnL-Logik (Endwert - Startkapital)
    var_pnl = var_end_value - capital
    es_pnl = es_end_value - capital

    return var_pnl, es_pnl

Monte-Carlo-Simulation wird hier für längere Horizonte genutzt, weil sie viele mögliche Zukunftspfade erzeugt und den Zinseszinseffekt über die Zeit sauber in den Endwerten abbildet. Modellbasis ist die geometrische Brownsche Bewegung mit IID-normalverteilten Logrenditen (Albrecht/Huggenberger 2015, S. 113 f.; Hull 2018, Kap. 14). Die Schwäche liegt in der Modellannahme: Verteilung, Mittelwert und Volatilität werden aus der Vergangenheit geschätzt und in die Zukunft fortgeschrieben.


In [19]:
def calculate_monte_carlo_risk(log_returns, capital, var_level, days, simulations=10000, black_swan=False, crash_data=None, monte_carlo_seed=monte_carlo_seed):
    
    clean_returns = log_returns.dropna()

    mu_log_1d = np.mean(clean_returns)
    sigma_log_1d = np.std(clean_returns)

    np.random.seed(monte_carlo_seed) #Setzen des Zufallsgenerators für Reproduzierbarkeit

    simulated_daily_returns = np.random.normal(mu_log_1d, sigma_log_1d, (days, simulations)) #Monte-Carlo Simulation der täglichen log Renditen

    if black_swan and crash_data is not None:
        crash_data_flat = np.asarray(crash_data).flatten() #Erzeugung eines Arrays
        crash_length = len(crash_data_flat)
        
        if days < crash_length:
            # Laufzeit ist kürzer als der Crash: Wir schneiden ein zufälliges Stück aus dem Crash heraus und fügen es in die Simulation ein
            slice_starts = np.random.randint(0, crash_length - days + 1, size=simulations)
            for s in range(simulations):
                start_idx_crash = slice_starts[s]
                simulated_daily_returns[:, s] = crash_data_flat[start_idx_crash : start_idx_crash + days]
        else:
            # Laufzeit ist länger als der Crash
            # Platzierung des Crashs zufällig innerhalb der Simulationsperiode (so dass er nicht immer am Anfang oder Ende liegt)
            crash_start_days = np.random.randint(0, max(1, days - int(crash_length/2)), size=simulations) #Random Starttag des Crashs 
            for s in range(simulations): #s iteriert über alle Simulationen, aktuelles s = Simulation
                start_idx = crash_start_days[s] #Startindex des Crashs in der Iteration s
                end_idx = min(days, start_idx + crash_length) #Endindex des Crashs, abhängig von der Länge der Simulationsperiode
                actual_crash_len = end_idx - start_idx #Tatsächliche Länge des Crashs, die in die Simulation eingefügt wird (kann kürzer sein als der gesamte Crash, wenn er am Ende der Simulationsperiode liegt)
                simulated_daily_returns[start_idx:end_idx, s] = crash_data_flat[:actual_crash_len] #Einfügen der Crash-Daten in die Simulation, Überschreibt die alten Daten 

            
    cumulative_log_returns = np.cumsum(simulated_daily_returns, axis=0)
    portfolio_paths = capital * np.exp(cumulative_log_returns) #
    
    final_values = portfolio_paths[-1, :]

    var_end_value = np.percentile(final_values, var_level * 100)
    tail_values = final_values[final_values <= var_end_value]
    es_end_value = np.mean(tail_values) if len(tail_values) > 0 else var_end_value

    var_PnL = var_end_value - capital #PnL = Profit and Loss, hier berechnet als Endwert - Anfangskapital, vorher var_pct * capital, aber hier mit aboluten Werten 
    es_PnL = es_end_value - capital

    return var_PnL, es_PnL, final_values, portfolio_paths

Bootstrapping simuliert zukünftige Portfoliowerte durch Ziehen mit Zurücklegen aus der historischen Renditeverteilung. Das ist für unser Projekt die pragmatische historische Langfristvariante. Die Stärke ist die Nähe zu beobachteten Renditen. Die Schwäche ist, dass neue Krisenmuster, Regimewechsel oder Zeitabhängigkeiten nicht explizit modelliert werden.

In [20]:
def calculate_bootstrap_risk(log_returns, capital, var_level, days, simulations=10000):
    clean_returns = log_returns.dropna()

    np.random.seed(bootstrap_seed)
    simulated_daily_returns = np.random.choice(clean_returns, size=(days, simulations), replace=True) #Bootstrap-Sampling der täglichen log Renditen
    cumulative_log_returns = np.sum(simulated_daily_returns, axis=0) 
    final_values = capital * np.exp(cumulative_log_returns)

    var_end_value = np.percentile(final_values, var_level * 100)
    tail_values = final_values[final_values <= var_end_value]
    es_end_value = np.mean(tail_values)

    var_PnL = var_end_value - capital
    es_PnL = es_end_value - capital

    return var_PnL, es_PnL

# Berechnung und Ausführung der Funktionen

## 1-Tages-Modelle

## Skalierung auf 1 Jahr, 5 und 10 Jahre bei 95 % und 99 %

Die drei Methoden laufen je nach Horizont **unterschiedlich stark auseinander**. Auf 1 Jahr trifft eine empirische Verteilung auf zwei parametrische, da entstehen die groessten Methodenunterschiede, vor allem im 99-%-Tail. Auf 5 und 10 Jahren glaettet die Aggregation Schiefe und Kurtosis, die Methodenwerte ruecken naeher zusammen. Wir lesen die Tabelle deshalb nicht als Punktwert, sondern als **Bandbreite**. Wie weit die drei Verfahren auseinanderliegen, sagt der Robustheitsblock weiter unten.

Die Kernergebnisse werden kompakt in einer Tabelle berichtet.

- **VaR** und **ES** werden in der P&L-Konvention berichtet.
- **Negative Werte** bedeuten Verlust.
- **Positive Werte** bedeuten, dass selbst das betrachtete Quantil noch über dem Anfangskapital liegt.


In [21]:
core_rows = []

for horizon_label, days in horizons.items():
    for confidence_label, alpha_level in confidence_levels.items():
        
        bhs_var, bhs_es = calculate_historical_risk(
            portfolio_returns_log, start_capital, alpha_level, days
        )
        
        boot_var, boot_es = calculate_bootstrap_risk(
            portfolio_returns_log, start_capital, alpha_level, days, simulations=bootstrap_simulations
        )

        g_var, g_es = calculate_gaussian_risk(
            portfolio_returns_discrete, start_capital, alpha_level, days
        )
        
        mc_var_norm, mc_es_norm, _, _ = calculate_monte_carlo_risk(
            portfolio_returns_log, start_capital, alpha_level, days, 
            simulations=monte_carlo_simulations, black_swan=False
        )
        
        mc_var_swan, mc_es_swan, _, _ = calculate_monte_carlo_risk(
            portfolio_returns_log, start_capital, alpha_level, days, 
            simulations=monte_carlo_simulations, black_swan=True, crash_data=dotcom_log_returns
        )

        core_rows.extend([
            {'Horizont': horizon_label, 'Handelstage': days, 'Konfidenzniveau': confidence_label, 'Methode': 'Historisch (BHS)', 'VaR ($)': bhs_var, 'ES ($)': bhs_es},
            {'Horizont': horizon_label, 'Handelstage': days, 'Konfidenzniveau': confidence_label, 'Methode': 'Historisch (Bootstrapping)', 'VaR ($)': boot_var, 'ES ($)': boot_es},
            {'Horizont': horizon_label, 'Handelstage': days, 'Konfidenzniveau': confidence_label, 'Methode': 'Gaußsch', 'VaR ($)': g_var, 'ES ($)': g_es},
            {'Horizont': horizon_label, 'Handelstage': days, 'Konfidenzniveau': confidence_label, 'Methode': 'Lognormal', 'VaR ($)': mc_var_norm, 'ES ($)': mc_es_norm},
            {'Horizont': horizon_label, 'Handelstage': days, 'Konfidenzniveau': confidence_label, 'Methode': 'Lognormal (MC DotCom-Crash)', 'VaR ($)': mc_var_swan, 'ES ($)': mc_es_swan},
        ])

core_results = pd.DataFrame(core_rows)

core_results['VaR (%)'] = core_results['VaR ($)'] / start_capital * 100
core_results['ES (%)'] = core_results['ES ($)'] / start_capital * 100

core_results_display = core_results.copy()
core_results_display[['VaR ($)', 'ES ($)', 'VaR (%)', 'ES (%)']] = core_results_display[['VaR ($)', 'ES ($)', 'VaR (%)', 'ES (%)']].round(2)

display(core_results_display)

,Horizont,Handelstage,Konfidenzniveau,Methode,VaR ($),ES ($),VaR (%),ES (%)
0,1 Jahr,252,95 %,Historisch (BHS),-17967.07,-31261.91,-17.97,-31.26
1,1 Jahr,252,95 %,Historisch (Bootstrapping),-15777.07,-24378.37,-15.78,-24.38
2,1 Jahr,252,95 %,Gaußsch,-10013.00,-21204.58,-10.01,-21.20
3,1 Jahr,252,95 %,Lognormal,-15482.02,-24595.07,-15.48,-24.60
4,1 Jahr,252,95 %,Lognormal (MC DotCom-Crash),-64502.40,-66957.00,-64.50,-66.96
5,1 Jahr,252,99 %,Historisch (BHS),-42907.10,-45950.50,-42.91,-45.95
6,1 Jahr,252,99 %,Historisch (Bootstrapping),-29945.51,-37003.48,-29.95,-37.00
7,1 Jahr,252,99 %,Gaußsch,-28265.55,-37341.45,-28.27,-37.34
8,1 Jahr,252,99 %,Lognormal,-30802.69,-36496.44,-30.80,-36.50
9,1 Jahr,252,99 %,Lognormal (MC DotCom-Crash),-68794.78,-69319.33,-68.79,-69.32


## Methodenvergleich

Hier stellen wir die drei Hauptmethoden direkt nebeneinander, für alle drei Horizonte und beide Konfidenzniveaus. So sieht man auf einen Blick, wo sie konvergieren und wo sie auseinanderlaufen. Die Black-Swan-Variante lassen wir bewusst raus, weil sie konzeptionell anders ist und im Black-Swan-Block separat behandelt wird.


In [22]:
# 1. Daten direkt übernehmen (keine Konsolidierung mehr nötig)
plot_data = core_results.copy()

# 2. Erweitertes Dictionary für alle 6 Traces
plot_methods = [
    'Historisch (BHS)', 
    'Historisch (Bootstrapping)', 
    'Gaußsch', 
    'Lognormal (MC)'
]

# Didaktische Farbwahl für die Präsentation
method_colors = {
    'Historisch (BHS)': 'rgb(133, 193, 233)',          # Hellblau
    'Historisch (Bootstrapping)': 'rgb(41, 128, 185)', # Dunkelblau
    'Gaußsch': 'rgb(231, 76, 60)',                     # Rot
    'Lognormal': 'rgb(35, 155, 86)'   # Dunkelgrün
}

fig_p2 = make_subplots(rows=1, cols=2, subplot_titles=("95 % Konfidenz", "99 % Konfidenz"), shared_yaxes=True)
horizon_order = list(horizons.keys())

for col_idx, conf_label in enumerate(confidence_levels.keys(), start=1):
    subset = plot_data[plot_data['Konfidenzniveau'] == conf_label]
    
    for method in plot_methods:
        method_data = subset[subset['Methode'] == method]
        if method_data.empty:
            continue
            
        method_data = method_data.set_index('Horizont').reindex(horizon_order).reset_index()
        
        fig_p2.add_trace(
            go.Bar(
                x=method_data['Horizont'],
                y=method_data['VaR ($)'],
                name=method,
                marker_color=method_colors[method],
                showlegend=(col_idx == 1),
                legendgroup=method 
            ),
            row=1, col=col_idx,
        )
        
    fig_p2.update_yaxes(title_text='VaR (USD)', row=1, col=col_idx)
    fig_p2.update_xaxes(title_text='Horizont', row=1, col=col_idx)

fig_p2.update_layout(
    template='plotly_dark',
    title='Methodenvergleich: Analytische Skalierung vs. Dynamische Simulation',
    barmode='group',
    height=550, # Leicht erhöht für die größere Legende
)

fig_p2.show()

## Methodenvergleich als Verteilungs-Overlay - 1 Jahr

Plot 2 zeigt die Punktwerte der Methoden nebeneinander. Hier zeigen wir die zugrundeliegenden Verteilungsannahmen direkt auf einer Achse: das historische Fenster hat fat tails, die Gauss-Annahme ist symmetrisch und schmal, die Lognormal-MC-Endwerte sind rechtsschief. Genau dieses Bild ist der didaktische Kern des Methodenvergleichs (Risk-Folien S. 22).


In [23]:
# Plot C: drei Verteilungen + 95-VaR-Linien auf einer Achse, P&L 1 Jahr
days_1y = horizons['1 Jahr']

# Historisch: rollierende 1J-P&L
rolling_log_C = portfolio_returns_log.rolling(window=days_1y).sum().dropna()
hist_pnl_C = ((np.exp(rolling_log_C) - 1) * start_capital).values

# Gauss: analytische Dichte über denselben x-Bereich
clean_disc = portfolio_returns_discrete.dropna()
mu_g = clean_disc.mean() * days_1y
sigma_g = clean_disc.std() * np.sqrt(days_1y)

# Lognormal MC: Endwerte ziehen, in PnL umrechnen
_, _, final_values_C, _ = calculate_monte_carlo_risk(
    portfolio_returns_log, start_capital, 0.05, days_1y,
    simulations=monte_carlo_simulations, black_swan=False,
)
mc_pnl_C = final_values_C - start_capital

# Gemeinsames x-Grid über die Vereinigung der drei Verteilungen
x_lo = float(min(hist_pnl_C.min(), mc_pnl_C.min(), mu_g - 4*sigma_g*start_capital))
x_hi = float(max(hist_pnl_C.max(), mc_pnl_C.max(), mu_g + 4*sigma_g*start_capital))
x_grid_C = np.linspace(x_lo, x_hi, 500)

# KDEs für Hist und MC
kde_hist = stats.gaussian_kde(hist_pnl_C)
kde_mc = stats.gaussian_kde(mc_pnl_C)
density_hist = kde_hist(x_grid_C)
density_mc = kde_mc(x_grid_C)

# Gauss-Dichte: x = r_disc * K, also r = x/K, dr/dx = 1/K
gauss_density = stats.norm.pdf(x_grid_C / start_capital, loc=mu_g, scale=sigma_g) / start_capital

# 95-VaR-Linien je Methode (alle in PnL-Konvention, negativ)
var_hist_C, _ = calculate_historical_risk(portfolio_returns_log, start_capital, 0.05, days_1y)
var_gauss_C, _ = calculate_gaussian_risk(portfolio_returns_discrete, start_capital, 0.05, days_1y)
var_mc_C = float(np.percentile(mc_pnl_C, 5))

fig_pC = go.Figure()
fig_pC.add_trace(go.Scatter(
    x=x_grid_C, y=density_hist, mode='lines',
    line=dict(color='rgb(133, 193, 233)', width=2.2),
    name='Historisch (BHS, rollierend)',
))
fig_pC.add_trace(go.Scatter(
    x=x_grid_C, y=gauss_density, mode='lines',
    line=dict(color='rgb(231, 76, 60)', width=2.2),
    name='Gaußsch',
))
fig_pC.add_trace(go.Scatter(
    x=x_grid_C, y=density_mc, mode='lines',
    line=dict(color='rgb(35, 155, 86)', width=2.2),
    name='Lognormal',
))

fig_pC.add_vline(x=var_hist_C, line=dict(color='rgb(52, 152, 219)', width=1.5, dash='dash'),
                 annotation_text=f"VaR Hist: {var_hist_C:,.0f}", annotation_position='top left')
fig_pC.add_vline(x=var_gauss_C, line=dict(color='rgb(231, 76, 60)', width=1.5, dash='dash'),
                 annotation_text=f"VaR Gauss: {var_gauss_C:,.0f}", annotation_position='top')
fig_pC.add_vline(x=var_mc_C, line=dict(color='rgb(46, 204, 113)', width=1.5, dash='dash'),
                 annotation_text=f"VaR MC: {var_mc_C:,.0f}", annotation_position='top right')

fig_pC.update_layout(
    template='plotly_dark',
    title='Methodenvergleich auf einer Achse - 1 Jahr',
    xaxis_title='Gewinn/Verlust (USD)',
    yaxis_title='Dichte',
    height=520,
)
fig_pC.show()


## Simulierte Wertentwicklungspfade

200 zufällig ausgewählte simulierte Pfade (aus 10.000 Simulationen) auf 5 Jahre. Das gibt ein Gefühl, wie breit Lognormal-MC streut.


In [24]:
# Monte-Carlo-Pfade für 5 Jahre ziehen
mc_var_5y, mc_es_5y, _, mc_paths_5y = calculate_monte_carlo_risk(
    portfolio_returns_log,
    start_capital,
    reference_alpha,
    horizons['5 Jahre'],
    simulations=monte_carlo_simulations,
    black_swan=False,
)

n_days, n_sims = mc_paths_5y.shape
days_axis = np.arange(n_days)

# Plot 7: 200 zufaellige Pfade als duenne, halbtransparente Linien
rng = np.random.default_rng(42)
sample_idx = rng.choice(n_sims, size=200, replace=False)

fig_p7 = go.Figure()
for s in sample_idx:
    fig_p7.add_trace(go.Scatter(
        x=days_axis, y=mc_paths_5y[:, s],
        mode='lines',
        line=dict(color='rgba(52, 152, 219, 0.15)', width=0.3),
        showlegend=False,
        hoverinfo='skip',
    ))

# Median-Pfad darübergelegt
median_path = np.median(mc_paths_5y, axis=1)
fig_p7.add_trace(go.Scatter(
    x=days_axis, y=median_path,
    mode='lines',
    line=dict(color='rgb(46, 204, 113)', width=2.5),
    name='Median-Pfad',
))
fig_p7.add_hline(y=start_capital, line=dict(color='rgb(231, 76, 60)', width=1, dash='dash'),
                 annotation_text=f"Startkapital ({start_capital:,.0f} USD)", annotation_position='bottom right')

fig_p7.update_layout(
    template='plotly_dark',
    title='200 von 10.000 simulierten Wertentwicklungspfaden - Lognormal Monte Carlo - 5 Jahre',
    xaxis_title='Handelstage',
    yaxis_title='Portfoliowert (USD)',
    height=520,
)
fig_p7.show()


## Quantilfächer für alle drei Horizonte

Plot 3 hat das Bild für 5 Jahre. Hier ziehen wir es konsequent auf den MVP-Horizont durch: 1 Jahr, 5 Jahre, 10 Jahre nebeneinander. Man sieht direkt, wie sich das 5/95-Band mit der Wurzel der Zeit aufweitet. Genau diese Aufweitung ist die Unsicherheit, die der Methodenvergleich quantifiziert.


In [25]:
# Plot B: drei MC-Laeufe (1J, 5J, 10J), Quantilfaecher 5/50/95 nebeneinander
# Hinweis: drei MC-Laeufe a 10.000 Sims sind machbar, aber spuerbar.
horizons_B = [('1 Jahr', horizons['1 Jahr']), ('5 Jahre', horizons['5 Jahre']), ('10 Jahre', horizons['10 Jahre'])]

fig_pB = make_subplots(rows=1, cols=3,
                       subplot_titles=('1 Jahr', '5 Jahre', '10 Jahre'),
                       shared_yaxes=False)

for col_idx, (label, n_days_B) in enumerate(horizons_B, start=1):
    _, _, _, paths_B = calculate_monte_carlo_risk(
        portfolio_returns_log, start_capital, 0.05, n_days_B,
        simulations=monte_carlo_simulations, black_swan=False,
    )
    days_axis_B = np.arange(paths_B.shape[0])
    q05_B = np.percentile(paths_B, 5, axis=1)
    q50_B = np.percentile(paths_B, 50, axis=1)
    q95_B = np.percentile(paths_B, 95, axis=1)

    show_leg = (col_idx == 1)
    fig_pB.add_trace(go.Scatter(
        x=days_axis_B, y=q05_B, mode='lines',
        line=dict(color='rgba(52, 152, 219, 0.0)', width=0),
        name='5 %-Quantil', showlegend=show_leg,
    ), row=1, col=col_idx)
    fig_pB.add_trace(go.Scatter(
        x=days_axis_B, y=q95_B, mode='lines',
        line=dict(color='rgba(52, 152, 219, 0.0)', width=0),
        fill='tonexty', fillcolor='rgba(52, 152, 219, 0.25)',
        name='95 %-Quantil', showlegend=show_leg,
    ), row=1, col=col_idx)
    fig_pB.add_trace(go.Scatter(
        x=days_axis_B, y=q50_B, mode='lines',
        line=dict(color='rgb(46, 204, 113)', width=2.2),
        name='Median (50 %)', showlegend=show_leg,
    ), row=1, col=col_idx)
    fig_pB.add_hline(y=start_capital, line=dict(color='rgb(231, 76, 60)', width=1, dash='dash'),
                     row=1, col=col_idx)
    fig_pB.update_xaxes(title_text='Handelstage', row=1, col=col_idx)
    fig_pB.update_yaxes(title_text='Portfoliowert (USD)', row=1, col=col_idx)

fig_pB.update_layout(
    template='plotly_dark',
    title='Quantilfächer 5/50/95 - Lognormal Monte Carlo - 1 / 5 / 10 Jahre',
    height=500,
)
fig_pB.show()


## Robustheitsindikator

Der Robustheitsindikator vergleicht die **VaR-Werte der drei Methoden** je Horizont und Konfidenzniveau.

Berechnet werden:
- **Absolute Spannweite** = größter VaR minus kleinster VaR
- **Relative Spannweite** = absolute Spannweite geteilt durch den Median der VaR-Werte

Die Einteilung **niedrig / mittel / hoch** ist hier eine **didaktische Heuristik**. Sie ist kein statistischer Test. Sie zeigt nur, wie stark die Methoden im Ergebnis auseinanderliegen.

Wichtig: Liegt der Median der VaR-Werte nahe null, kann die relative Spannweite sehr groß werden. Das ist dann ein Hinweis auf hohe Modellabhängigkeit rund um die Gewinn-/Verlustgrenze und kein Beweis für einen Rechenfehler.

In [26]:
robustness_rows = []
core_results_base = core_results[core_results['Methode'] != 'Lognormal (DotCom-Crash)']

for (horizon_label, days, confidence_label), subset in core_results_base.groupby(['Horizont', 'Handelstage', 'Konfidenzniveau']):
    var_values = subset['VaR ($)'].to_numpy(dtype=float)
    absolute_range = float(var_values.max() - var_values.min())
    median_var = float(np.median(var_values))
    denominator = max(abs(median_var), 1e-9)
    relative_range = absolute_range / denominator

    if relative_range < 0.15:
        robustness_class = 'niedrig'
    elif relative_range <= 0.35:
        robustness_class = 'mittel'
    else:
        robustness_class = 'hoch'

    robustness_rows.append({
        'Horizont': horizon_label,
        'Handelstage': days,
        'Konfidenzniveau': confidence_label,
        'Absolute Spannweite VaR ($)': absolute_range,
        'Relative Spannweite VaR': relative_range,
        'Relative Spannweite VaR (%)': relative_range * 100,
        'Robustheitsklasse': robustness_class,
    })

robustness_summary = pd.DataFrame(robustness_rows).sort_values(['Handelstage', 'Konfidenzniveau']).reset_index(drop=True)
robustness_summary[['Absolute Spannweite VaR ($)', 'Relative Spannweite VaR', 'Relative Spannweite VaR (%)']] = robustness_summary[['Absolute Spannweite VaR ($)', 'Relative Spannweite VaR', 'Relative Spannweite VaR (%)']].round(2)

display(robustness_summary)

,Horizont,Handelstage,Konfidenzniveau,Absolute Spannweite VaR ($),Relative Spannweite VaR,Relative Spannweite VaR (%),Robustheitsklasse
0,1 Jahr,252,95 %,54489.40,3.45,345.37,hoch
1,1 Jahr,252,99 %,40529.23,1.32,131.58,hoch
2,5 Jahre,1260,95 %,239085.91,5.80,580.08,hoch
3,5 Jahre,1260,99 %,210464.56,27.33,2732.79,hoch
4,10 Jahre,2520,95 %,1273180.46,4.94,494.41,hoch
5,10 Jahre,2520,99 %,1177738.88,11.12,1111.99,hoch


# Black-Swan-Block

In [27]:
def plot_black_swan_comparison(paths_normal, paths_swan, start_capital=100000, title="Median-Vergleich: Normal vs. Black Swan"):
    """
    Legt die Median-Rendite der normalen Simulation und der Black-Swan-Simulation übereinander.
    """
    days = paths_normal.shape[0]
    
    # Mediane berechnen (quer über alle 10.000 Simulationen)
    median_normal = np.median(paths_normal, axis=1)
    median_swan = np.median(paths_swan, axis=1)
    
    # Tag 0 (Startkapital) einfügen
    x_axis = np.arange(1, days + 1)
    x_axis = np.insert(x_axis, 0, 0)
    median_normal = np.insert(median_normal, 0, start_capital)
    median_swan = np.insert(median_swan, 0, start_capital)
    
    fig = go.Figure()
    
    # Normale Linie (Blau)
    fig.add_trace(go.Scatter(
        x=x_axis, y=median_normal,
        mode='lines', line=dict(color='rgb(52, 152, 219)', width=3),
        name='Median (Ohne Crash)'
    ))
    
    # Black Swan Linie (Rot, gestrichelt für Dramatik)
    fig.add_trace(go.Scatter(
        x=x_axis, y=median_swan,
        mode='lines', line=dict(color='rgb(231, 76, 60)', width=3, dash='dash'),
        name='Median (Mit DotCom-Crash)'
    ))
    
    # Startkapital als Referenzlinie
    fig.add_hline(
        y=start_capital, 
        line_dash="dot", line_color="rgba(255, 255, 255, 0.3)", 
        annotation_text="Startkapital"
    )
    
    fig.update_layout(
        title=title, xaxis_title="Handelstage", yaxis_title="Portfolio-Wert ($)",
        template="plotly_dark", hovermode="x unified",
        margin=dict(l=20, r=20, t=50, b=20),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    
    return fig

Wir vergleichen den Median-Pfad einer normalen Monte-Carlo-Simulation mit einer Variante, in der ein historischer Crash-Verlauf (DotCom) zufällig in die Pfade eingewoben wird. So sieht man auf einen Blick, wie stark ein solcher Schock den typischen Pfad nach unten zieht.

In [28]:
# Pfade einmal mit und einmal ohne Crash ziehen, dann vergleichen.
_, _, _, paths_normal = calculate_monte_carlo_risk(
    portfolio_returns_log,
    start_capital,
    reference_alpha,
    horizons['1 Jahr'],
    simulations=monte_carlo_simulations,
    black_swan=False,
)

_, _, _, paths_swan = calculate_monte_carlo_risk(
    portfolio_returns_log,
    start_capital,
    reference_alpha,
    horizons['1 Jahr'],
    simulations=monte_carlo_simulations,
    black_swan=True,
    crash_data=dotcom_log_returns,
)

fig_swan = plot_black_swan_comparison(
    paths_normal=paths_normal,
    paths_swan=paths_swan,
    start_capital=start_capital,
    title='Median-Vergleich: Normal vs. DotCom-Crash (1 Jahr)',
)
fig_swan.show()


In [29]:
# Plot B: drei MC-Laeufe (1J, 5J, 10J), Quantilfaecher 5/50/95 nebeneinander #Für Blackswan-Analyse

horizons_B = [('1 Jahr', horizons['1 Jahr']), ('5 Jahre', horizons['5 Jahre']), ('10 Jahre', horizons['10 Jahre'])]

fig_pB = make_subplots(rows=1, cols=3,
                       subplot_titles=('1 Jahr', '5 Jahre', '10 Jahre'),
                       shared_yaxes=False)

for col_idx, (label, n_days_B) in enumerate(horizons_B, start=1):
    _, _, _, paths_B = calculate_monte_carlo_risk(
        portfolio_returns_log, start_capital, 0.05, n_days_B,
        simulations=monte_carlo_simulations, black_swan=True, crash_data=dotcom_log_returns,
    )
    days_axis_B = np.arange(paths_B.shape[0])
    q05_B = np.percentile(paths_B, 5, axis=1)
    q50_B = np.percentile(paths_B, 50, axis=1)
    q95_B = np.percentile(paths_B, 95, axis=1)

    show_leg = (col_idx == 1)
    fig_pB.add_trace(go.Scatter(
        x=days_axis_B, y=q05_B, mode='lines',
        line=dict(color='rgba(52, 152, 219, 0.0)', width=0),
        name='5 %-Quantil', showlegend=show_leg,
    ), row=1, col=col_idx)
    fig_pB.add_trace(go.Scatter(
        x=days_axis_B, y=q95_B, mode='lines',
        line=dict(color='rgba(52, 152, 219, 0.0)', width=0),
        fill='tonexty', fillcolor='rgba(52, 152, 219, 0.25)',
        name='95 %-Quantil', showlegend=show_leg,
    ), row=1, col=col_idx)
    fig_pB.add_trace(go.Scatter(
        x=days_axis_B, y=q50_B, mode='lines',
        line=dict(color='rgb(46, 204, 113)', width=2.2),
        name='Median (50 %)', showlegend=show_leg,
    ), row=1, col=col_idx)
    fig_pB.add_hline(y=start_capital, line=dict(color='rgb(231, 76, 60)', width=1, dash='dash'),
                     row=1, col=col_idx)
    fig_pB.update_xaxes(title_text='Handelstage', row=1, col=col_idx)
    fig_pB.update_yaxes(title_text='Portfoliowert (USD)', row=1, col=col_idx)

fig_pB.update_layout(
    template='plotly_dark',
    title='Quantilfächer 5/50/95 - Lognormal Monte Carlo - 1 / 5 / 10 Jahre',
    height=500,
)
fig_pB.show()


# KPIs 

In [30]:
def calculate_performance_kpis(portfolio_returns, benchmark_world_data, benchmark_risk_free_data):
    
    # Aufbereitung der Daten für die Berechnung der KPIs 
    benchmark_returns = benchmark_world_data.pct_change().dropna().squeeze() #.squeeze() "quetscht" die DataFrame-Struktur, damit wir eine Series haben
    risk_free_daily = (benchmark_risk_free_data.dropna() / 100 / 252).squeeze() #/100 um auf Prozente zu kommen 

    # Daten synchronisieren (gleiche Handelstage usw. hier als für flexibilität bei in USA gehandelten Assets gleich)
    aligned_kpi_data = pd.concat([
        portfolio_returns.squeeze().rename('Portfolio'), 
        benchmark_returns.rename('Market'), 
        risk_free_daily.rename('RiskFree')
    ], axis=1, sort=False).dropna()

    port_ret = aligned_kpi_data['Portfolio'] #Portfolio-Renditen (hier Returns, also ret)
    mkt_ret = aligned_kpi_data['Market']
    rf_ret = aligned_kpi_data['RiskFree']

    #Berechnung der KPIs (Tagesbasis und Annualisiert)

    # Beta berechnen (Kovarianz Portfolio&Markt / Varianz Markt)
    cov_matrix = np.cov(port_ret, mkt_ret)
    beta = cov_matrix[0, 1] / cov_matrix[1, 1] #Beta als Maß wie stark Portfolio im Vergleich zum Markt schwankt 

    # Erwartungswerte (Durchschnitte)
    mu_rp_daily = np.mean(port_ret) #Erwartungswert der Portfolio-Renditen
    mu_rm_daily = np.mean(mkt_ret) #Erwartungswert der Markt-Renditen
    mu_rf_daily = np.mean(rf_ret) #Erwartungswert der risikofreien Renditen
    sigma_rp_daily = np.std(port_ret) #Standardabweichung der Portfolio-Renditen

    # Sharpe Ratio
    sharpe_daily = (mu_rp_daily - mu_rf_daily) / sigma_rp_daily
    sharpe_ann = sharpe_daily * np.sqrt(252)

    # Roy's Safety First Ratio (Nutzung vermeitlich Risikofreier Zins, fraglich ob USA-Staatsanleihen als risikofreier Zins gilt)
    rsf_daily = (mu_rp_daily - mu_rm_daily) / sigma_rp_daily
    rsf_ann = rsf_daily * np.sqrt(252)

    # Treynor Ratio (direkt Jahr dan Beta annualisiert ist)
    mu_rp_anno = mu_rp_daily * 252
    mu_rf_anno = mu_rf_daily * 252
    treynor_ann = (mu_rp_anno - mu_rf_anno) / beta

    # --- 3. Ausgabe ---
    print(f"\n{' PERFORMANCE KPIS (Annualisiert) ':=^75}")
    print(f"Portfolio Beta:      {beta:.2f}")
    print(f"Sharpe Ratio:        {sharpe_ann:.2f}")
    print(f"Roy's Safety First:  {rsf_ann:.2f}")
    print(f"Treynor Ratio:       {treynor_ann:.4f}")
    
    # Rückgabe der berechneten Werte als Dictionary für die spätere Weiterverwendung
    return {
        "Beta": beta,
        "Sharpe_Ratio": sharpe_ann,
        "Roys_Safety_First": rsf_ann,
        "Treynor_Ratio": treynor_ann
    }


In [31]:
kpi_results = calculate_performance_kpis(
    portfolio_returns=portfolio_returns_discrete, 
    benchmark_world_data=benchmark_world_data, 
    benchmark_risk_free_data=benchmark_risk_free_data
)


===================== PERFORMANCE KPIS (Annualisiert) =====================
Portfolio Beta:      1.28
Sharpe Ratio:        1.16
Roy's Safety First:  0.81
Treynor Ratio:       0.2425
